> **Portfolio summary:** Visual RAG over a PDF financial report with LlamaIndex and the Qwen2.5-VL multimodal model (code only; needs a GPU).
>
> [Project README](README.md) · [Back to the portfolio](../README.md)

This notebook demonstrates an end-to-end Visual RAG (Retrieval Augmented Generation) pipeline. It uses LlamaIndex and the Qwen2.5-VL large multimodal model to answer complex questions by combining semantic search over text with visual analysis of document images from a PDF financial report. The process involves loading an embedding model, loading a Visual Large Language Model (VLM), setting up LlamaIndex, loading a PDF document into a vector index, and defining a function query_visual_rag to perform the RAG process. This function retrieves relevant text, locates the corresponding page in the PDF, renders the page as an image, and then uses the VLM to answer questions based on both the text and the image.

In [ ]:
!pip install -qU Pillow
print("Pillow upgraded")

###Step 1: Verify GPU and install dependencies




In [ ]:
import torch

if torch.cuda.is_available():
  gpu_name = torch.cuda.get_device_name(0)
  vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
  print(f" GPU detected: {gpu_name} {vram_gb:.1f} GB VRAM")
else:
  print("\u274c NO GPU DETECTED!")
  print("   Go to Runtime \u2192 Change runtime type \u2192 T4 GPU")
  print("   Running on CPU will make each query take 5-10 minutes.")
  raise RuntimeError("GPU required. Please change runtime type to T4 GPU and re-run.")

!pip install -qU transformers accelerate
!pip install -qU llama-index-core llama-index-readers-file llama-index-embeddings-huggingface pypdf
!pip install -qU pdf2image
!apt-get install -q poppler-utils

print("\u2705 Installation Complete.")

### Step 2: Load embedding model

In [ ]:
import os
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model loaded.")

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

VLM_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

print(f" Loading {VLM_MODEL_ID}...")
print(f"   Device: {'CUDA - ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (WARNING: will be very slow)'}")


vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda"
)
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)

# Confirm where the model actually landed
model_device = next(vlm_model.parameters()).device
print(f"\u2705 VLM loaded on: {model_device}")
if str(model_device) == 'cpu':
    print("\u26a0\ufe0f  WARNING: Model is on CPU. Each query will take 5-10 minutes!")

In [ ]:
Settings.embed_model = embed_model
Settings.llm = None
print("\u2705 LlamaIndex settings configured.")

In [ ]:
PDF_FILE = "AstraZeneca-Q4-2025-earnings.pdf"

print(f"\U0001f4da Loading {PDF_FILE}...")
documents = SimpleDirectoryReader(input_files=[PDF_FILE]).load_data()

print(f"   Loaded {len(documents)} pages.")
print(f"   Sample Metadata: {documents[0].metadata}")

In [ ]:
print(" Building Vector Index...")
index = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(similarity_top_k=1)
print(" Index Ready!")

In [ ]:
from pypdf import PdfReader, PdfWriter
from pdf2image import convert_from_path
from PIL import Image
import torch

def query_visual_rag(query_text, max_new_tokens=256):
    print(f"\n\U0001f50e Querying LlamaIndex for: '{query_text}'...")

    # Warn if model is on CPU before spending time on generation.
    model_device = next(vlm_model.parameters()).device
    if str(model_device) == 'cpu':
        print("\u26a0\ufe0f  WARNING: VLM is running on CPU. This will be very slow (5-10 min).")
        print("   Change runtime to T4 GPU and restart for fast inference.")

    # 1. RETRIEVE: semantic search over page text
    nodes = retriever.retrieve(query_text)
    if not nodes:
        return "\u274c No relevant information found in the index."

    # 2. LOCATE: get page number from metadata
    best_node = nodes[0]
    page_label = best_node.metadata.get('page_label')
    page_index = int(page_label) - 1 if page_label else 0

    print(f"\U0001f4cd Found answer on Page {page_label} (Score: {best_node.score:.4f})")
    print(f"   Context Snippet: {best_node.text[:100]}...")

    # 3. RENDER: convert that PDF page to an image
    print("\U0001f5bc\ufe0f  Rendering page as image...")
    pages = convert_from_path(
        PDF_FILE,
        first_page=page_index + 1,
        last_page=page_index + 1,
        dpi=150
    )
    page_image = pages[0]

    # 4. VISION: build the prompt and run Qwen2.5-VL
    print(f"\U0001f680 Sending page image to Qwen2.5-VL (max {max_new_tokens} tokens)...")
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": page_image},
                {"type": "text", "text": (
                    f"You are an expert financial analyst reviewing Page {page_label} "
                    # Corrected report name — was 'JLR Annual Report'
                    "of the Goldman Sachs Q4 2024 Earnings Report. "
                    "Answer the following question based on the text, tables, and "
                    "charts visible on this page. "
                    "If the answer involves a chart, describe the visual trend.\n\n"
                    f"Question: {query_text}"
                )}
            ]
        }
    ]

    text_prompt = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = vlm_processor(
        text=[text_prompt],
        images=[page_image],
        return_tensors="pt"
    ).to(vlm_model.device)

    with torch.no_grad():
        output_ids = vlm_model.generate(
            **inputs,
            # Reduced max_new_tokens from 512 to 256.
            # 512 tokens on T4 GPU takes ~15-20 sec; on CPU it takes ~5-10 MIN.
            # 256 tokens covers all typical financial QA answers in ~8-10 sec on GPU.
            # Caller can pass max_new_tokens=512 if longer answers are needed.
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,   # Remove temperature when do_sample=False
            top_p=None,         # to suppress the 'not valid' warning
        )

    # Decode only the newly generated tokens
    generated = output_ids[:, inputs['input_ids'].shape[1]:]
    answer = vlm_processor.batch_decode(generated, skip_special_tokens=True)[0]
    return answer

print("\u2705 query_visual_rag() function defined and ready.")

In [ ]:
from IPython.display import display, Markdown

q1 = "What were AstraZeneca's total Product Sales and Alliance Revenue for FY 2025, and how did each change compared to FY 2024?"
display(Markdown(f"### Q1: {q1}"))
display(Markdown(query_visual_rag(q1)))

In [ ]:
q2 = " Which geographic region had the highest Total Revenue growth in FY 2025, and what was the growth rate at constant exchange rates "
display(Markdown(f"### Q2: {q2}"))
display(Markdown(query_visual_rag(q2)))

In [ ]:
q3 = " Which medicines received regulatory approvals in the US between November 2025 and February 2026, and for what indications ? "
display(Markdown(f"### Q3: {q3}"))
display(Markdown(query_visual_rag(q3)))